## Notebook Overview

This notebook demonstrates a system for disease prediction and clinical recommendations.

**Data Download and Preparation**

In [ ]:
# 1. Download and unzip dataset
!kaggle datasets download -d behzadhassan/sympscan-symptomps-to-disease
!unzip -q sympscan-symptomps-to-disease.zip -d sympscan_data

# 2. Inspect directory files
import os
print("Files in dataset:", os.listdir('sympscan_data'))

Dataset URL: https://www.kaggle.com/datasets/behzadhassan/sympscan-symptomps-to-disease
License(s): CC0-1.0
sympscan-symptomps-to-disease.zip: Skipping, found more recently modified local copy (use --force to force download)
replace sympscan_data/Diseases_and_Symptoms_dataset.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: Files in dataset: ['description.csv', 'precautions.csv', 'medications.csv', 'diets.csv', 'workout.csv', 'Diseases_and_Symptoms_dataset.csv']


In [ ]:
import pandas as pd
import numpy as np

# Load files into DataFrames
diseases_df = pd.read_csv('sympscan_data/Diseases_and_Symptoms_dataset.csv')
medications_df = pd.read_csv('sympscan_data/medications.csv')
precautions_df = pd.read_csv('sympscan_data/precautions.csv')
diets_df = pd.read_csv('sympscan_data/diets.csv')
workout_df = pd.read_csv('sympscan_data/workout.csv')
description_df = pd.read_csv('sympscan_data/description.csv')

print("Main Dataset Shape:", diseases_df.shape)
print("\nColumns in Main Dataset:", diseases_df.columns.tolist()[:10])
print("\nDiseases represented:", diseases_df['diseases'].nunique() if 'diseases' in diseases_df.columns else "Check column name")

Main Dataset Shape: (96088, 231)

Columns in Main Dataset: ['diseases', 'anxiety and nervousness', 'depression', 'shortness of breath', 'depressive or psychotic symptoms', 'sharp chest pain', 'dizziness', 'insomnia', 'abnormal involuntary movements', 'chest tightness']

Diseases represented: 100


In [ ]:
diseases_df.head()

,diseases,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,...,sneezing,leg weakness,hysterical behavior,arm lump or mass,bleeding gums,pain in gums,diaper rash,hesitancy,back stiffness or tightness,low urine output
0,panic disorder,1,0,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,panic disorder,0,0,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,panic disorder,1,1,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,panic disorder,1,0,0,1,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,panic disorder,1,1,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
medications_df.head()

,Disease,Medication
0,Panic disorder,"['SSRIs (e.g., Sertraline, Fluoxetine)', 'Benz..."
1,Vaginitis,"['Metronidazole', 'Clindamycin', 'Fluconazole'..."
2,Problem during pregnancy,"['Prenatal vitamins', 'Iron supplements', 'Ant..."
3,Acute pancreatitis,"['IV fluids', 'Pain relievers (e.g., Morphine)..."
4,Asthma,"['Inhaled corticosteroids (e.g., Fluticasone)'..."


In [ ]:
precautions_df.head()

,Disease,Precaution_1,Precaution_2,Precaution_3,Precaution_4
0,Panic disorder,Practice deep breathing,Avoid caffeine,Follow therapy plan,Seek support from loved ones
1,Vaginitis,Wear breathable cotton underwear,Avoid douching,Maintain genital hygiene,Avoid scented hygiene products
2,Problem during pregnancy,Attend regular prenatal visits,Avoid alcohol and smoking,Eat a balanced diet,Get adequate rest
3,Acute pancreatitis,Avoid alcohol,Eat a low-fat diet,Stay hydrated,Follow doctor's advice strictly
4,Asthma,Avoid known triggers,Use inhaler as prescribed,Monitor peak flow,Keep emergency inhaler handy


In [ ]:
diets_df.head()

,Disease,Diet
0,Panic Disorder,"['Magnesium-rich foods (spinach, pumpkin seeds..."
1,Vaginitis,"['Probiotics (yogurt, kefir, sauerkraut)', 'Lo..."
2,Problem During Pregnancy,"['Prenatal vitamins (consult doctor)', 'Iron-r..."
3,Acute Pancreatitis,"['Low-fat foods (boiled vegetables, lean chick..."
4,Asthma,"['Anti-inflammatory foods (blueberries, kale, ..."


In [ ]:
workout_df.head()

,Disease,Workouts
0,Panic disorder,"[""Deep breathing exercises: Calm your mind by ..."
1,Vaginitis,"[""Pelvic floor exercises: Strengthen pelvic mu..."
2,Problem during pregnancy,"[""Prenatal yoga: Gentle stretches safe for pre..."
3,Acute pancreatitis,"[""Avoid heavy lifting: Prevent strain on pancr..."
4,Asthma,"[""Breathing exercises: Improve lung function"",..."


In [ ]:
description_df.head()

,Disease,Description
0,Panic disorder,Panic disorder is a mental health condition ma...
1,Vaginitis,Vaginitis is inflammation of the vaginal tissu...
2,Problem during pregnancy,Problems during pregnancy refer to medical com...
3,Acute pancreatitis,Acute pancreatitis is a sudden inflammation of...
4,Asthma,Asthma is a chronic inflammatory disease of th...


**Multi-Class Classification Model**

A Random Forest Classifier is trained to predict diseases based on symptoms. The process includes feature selection, one-hot encoding of categorical features, label encoding for diseases, and splitting data into training and testing sets. The model's performance is evaluated using accuracy and a classification report.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Separate Features (X) and Target (y)
target_col = 'diseases'
X = diseases_df.drop(columns=[target_col])
y = diseases_df[target_col]

In [ ]:
# 2. Convert categorical text features (if any) to one-hot encoding
if (X.dtypes == 'object').any():
    X = pd.get_dummies(X, drop_first=True)

# 3. Encode Disease Labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# 4. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# 5. Train Random Forest Classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# 6. Evaluate Model
y_pred = clf.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

Accuracy: 87.00%

                                              precision    recall  f1-score   support

                           actinic keratosis       0.78      0.73      0.76       162
                         acute bronchiolitis       0.92      0.89      0.90       241
                            acute bronchitis       0.74      0.72      0.73       243
                          acute bronchospasm       0.55      0.62      0.58       162
                         acute kidney injury       0.97      0.99      0.98       162
                          acute pancreatitis       0.90      0.84      0.87       241
                             acute sinusitis       0.85      0.85      0.85       166
                                     allergy       0.95      0.98      0.96       161
                                      angina       0.94      0.98      0.96       165
                                     anxiety       0.95      0.96      0.96       240
                                app

In [ ]:
import pandas as pd
import numpy as np

def predict_disease_from_binary_symptoms(user_symptoms, model, feature_columns, label_encoder):
    """
    Predicts disease given a list of active symptom strings.
    """
    # 1. Initialize a feature vector of zeros with exact training columns
    input_vector = pd.DataFrame(0, index=[0], columns=feature_columns)

    # 2. Set user's active symptoms to 1 (handling case sensitivity & formatting)
    matched_symptoms = []
    for symptom in user_symptoms:
        clean_symptom = symptom.strip().lower().replace(" ", "_")
        for col in feature_columns:
            if col.lower().replace(" ", "_") == clean_symptom:
                input_vector.at[0, col] = 1
                matched_symptoms.append(col)

    if not matched_symptoms:
        return "No valid symptoms matched dataset features.", []

    # 3. Predict class integer and decode to disease name
    pred_encoded = model.predict(input_vector)[0]
    predicted_disease = label_encoder.inverse_transform([pred_encoded])[0]

    # 4. Get probability distribution across top candidate diseases
    probabilities = model.predict_proba(input_vector)[0]
    top_indices = np.argsort(probabilities)[::-1][:3]
    top_candidates = [
        (label_encoder.classes_[idx], round(probabilities[idx] * 100, 2))
        for idx in top_indices if probabilities[idx] > 0
    ]

    return predicted_disease, top_candidates

# --- Example Usage ---
# Assuming `clf`, `X.columns`, and `label_encoder` are from your trained model:
user_input = ['itching', 'skin_rash', 'nodal_skin_eruptions']

primary_disease, top_3_probs = predict_disease_from_binary_symptoms(
    user_symptoms=user_input,
    model=clf,
    feature_columns=X.columns,
    label_encoder=label_encoder
)

print(f"Top Predicted Disease: {primary_disease}")
print("\nTop Probabilities:")
for disease, prob in top_3_probs:
    print(f"- {disease}: {prob}%")

Top Predicted Disease: eczema

Top Probabilities:
- eczema: 59.4%
- allergy: 10.5%
- diaper rash: 6.0%


**Rule-Based Inference Engine**

A custom inference engine is developed to suggest potential diseases based on a list of user-provided symptoms. This engine includes functionalities for cleaning text, identifying known symptoms within the dataset, providing alerts for unlisted symptoms.

In [ ]:
import pandas as pd
import numpy as np
import math

def clean_text(text):
    if pd.isna(text):
        return ""
    return str(text).strip().lower().replace(' ', '_').replace('-', '_')

def get_dataset_symptom_universe(dataset, target_col, feature_cols, is_binary_matrix):
    """
    Extracts every valid symptom present in the dataset.
    """
    universe = set()
    if is_binary_matrix:
        for col in feature_cols:
            if (dataset[col] > 0).any():
                universe.add(clean_text(col))
    else:
        for col in feature_cols:
            for val in dataset[col].dropna().unique():
                c_val = clean_text(val)
                if c_val and c_val not in ['none', 'nan', '0']:
                    universe.add(c_val)
    return universe

def find_resembling_symptoms(unmatched_input, dataset_symptom_universe):
    """
    Finds resembling symptoms in the dataset based on shared tokens and character substrings.
    """
    resembling = set()
    input_tokens = set(unmatched_input.split('_'))

    for d_sym in dataset_symptom_universe:
        # Token overlap check (e.g., 'skin' matches 'skin_rash')
        d_tokens = set(d_sym.split('_'))
        if input_tokens.intersection(d_tokens):
            resembling.add(d_sym)
            continue

        # Substring/partial match check
        if unmatched_input in d_sym or d_sym in unmatched_input:
            resembling.add(d_sym)

    return sorted(list(resembling))

def fuzzy_match_symptoms(user_symptoms, disease_symptoms):
    """
    Finds matching disease symptoms for a user's input list.
    """
    matched_symptoms = set()

    for u_clean in user_symptoms:
        if not u_clean:
            continue

        # Direct exact match
        if u_clean in disease_symptoms:
            matched_symptoms.add(u_clean)
            continue

        # Substring / partial match search
        for d_sym in disease_symptoms:
            if u_clean in d_sym or d_sym in u_clean:
                matched_symptoms.add(d_sym)

    return matched_symptoms

def rule_based_inference(user_symptoms, dataset, metric='overlap', top_n=5):
    """
    Robust inference engine with symptom universe validation, unlisted symptom alerts,
    and resembling symptom suggestions.
    """
    # 1. Detect target column and feature columns
    target_col = 'Disease' if 'Disease' in dataset.columns else dataset.columns[0]
    feature_cols = [col for col in dataset.columns if col != target_col]

    # 2. Determine dataset format
    sample_val = dataset[feature_cols[0]].dropna().iloc[0]
    is_binary_matrix = isinstance(sample_val, (int, float, bool, np.number))

    # 3. Build global dataset symptom universe
    dataset_symptom_universe = get_dataset_symptom_universe(dataset, target_col, feature_cols, is_binary_matrix)

    # 4. Clean user inputs and check against dataset universe
    cleaned_user_inputs = [clean_text(s) for s in user_symptoms if s]

    unlisted_symptoms_info = []
    valid_user_inputs = []

    # 5. Run diagnosis inference
    results = []

    for disease_name, group in dataset.groupby(target_col):
        disease_symptoms = set()

        if is_binary_matrix:
            for col in feature_cols:
                if (group[col] > 0).any():
                    disease_symptoms.add(clean_text(col))
        else:
            for col in feature_cols:
                for sym in group[col].dropna().unique():
                    c_sym = clean_text(sym)
                    if c_sym and c_sym not in ['none', 'nan', '0']:
                        disease_symptoms.add(c_sym)

        matched = fuzzy_match_symptoms(valid_user_inputs, disease_symptoms)
        intersection_len = len(matched)

        if intersection_len == 0:
            continue

        size_user = max(len(valid_user_inputs), 1)
        size_disease = max(len(disease_symptoms), 1)

        if metric == 'overlap':
            score = intersection_len / min(size_user, size_disease)
        elif metric == 'cosine':
            score = intersection_len / math.sqrt(size_user * size_disease)
        elif metric == 'jaccard':
            union_len = len(set(valid_user_inputs).union(disease_symptoms))
            score = intersection_len / union_len if union_len > 0 else 0
        else:
            score = intersection_len / size_user

        results.append({
            'disease': disease_name,
            'confidence': score,
            'matched_symptoms': list(matched),
            'disease_symptom_count': size_disease
        })

    sorted_results = sorted(results, key=lambda x: x['confidence'], reverse=True)[:top_n]

    return {
        'diagnoses': sorted_results,
        'unlisted_symptoms': unlisted_symptoms_info
    }

In [ ]:
# Test with a mix of valid, partially matching, and completely missing symptoms
sample_input = ['itching', 'skin rash']

response = rule_based_inference(sample_input, diseases_df, metric='overlap')

print("\n--- TOP DIAGNOSES ---")
for res in response['diagnoses']:
    print(f"Disease: {res['disease']} | Confidence: {res['confidence']:.2f} | Matched: {res['matched_symptoms']}")

for res in response['unlisted_symptoms']:
    print(f"Unlisted Symptoms: {res['input']}")


--- TOP DIAGNOSES ---
Disease: actinic keratosis | Confidence: 1.00 | Matched: ['itching_of_skin', 'skin_rash']
Disease: allergy | Confidence: 1.00 | Matched: ['itching_of_skin', 'skin_rash']
Disease: contact dermatitis | Confidence: 1.00 | Matched: ['itching_of_skin', 'skin_rash']
Disease: drug reaction | Confidence: 1.00 | Matched: ['itching_of_skin', 'skin_rash']
Disease: eczema | Confidence: 1.00 | Matched: ['itching_of_skin', 'skin_rash']


**Multi-Output Recommender System**

A ClinicalRecommenderSystem class is implemented to provide comprehensive recommendations for a given disease. It leverages the loaded DataFrames to fetch disease descriptions, medications, precautions, dietary plans, and recommended workouts

In [ ]:
import pandas as pd

class ClinicalRecommenderSystem:
    def __init__(self, desc_df, med_df, prec_df, diet_df, work_df):
        self.desc_df = desc_df
        self.med_df = med_df
        self.prec_df = prec_df
        self.diet_df = diet_df
        self.work_df = work_df

        # Pre-process all DataFrames to normalize disease names for easy lookup
        self._normalize_dataframes()

    def _clean_string(self, val):
        """Standardizes strings by replacing underscores, lowercasing, and stripping whitespace."""
        if pd.isna(val):
            return ""
        return str(val).lower().replace("_", " ").strip()

    def _normalize_dataframes(self):
        """Finds the disease column in each dataframe and creates a normalized '_clean_disease' column."""
        dfs = [self.desc_df, self.med_df, self.prec_df, self.diet_df, self.work_df]
        for df in dfs:
            # Locate disease column regardless of case or naming convention
            disease_col = next((c for c in df.columns if c.lower() in ['disease', 'diseases']), None)
            if disease_col:
                df['_clean_disease'] = df[disease_col].apply(self._clean_string)

    def _extract_all_values(self, df, clean_disease_name):
        """Extracts all non-disease text entries from a row, supporting both wide and long CSV formats."""
        matched_rows = df[df['_clean_disease'] == clean_disease_name]
        if matched_rows.empty:
            return []

        # Exclude original disease column, clean column, and index-like columns
        ignore_cols = ['_clean_disease', 'disease', 'diseases', 'unnamed: 0', 'index']
        val_cols = [c for c in matched_rows.columns if c.lower() not in ignore_cols]

        extracted_items = []
        for _, row in matched_rows.iterrows():
            for col in val_cols:
                val = row[col]
                if pd.notna(val) and str(val).strip() != "":
                    # Parse evaluated list strings if present (e.g., "['drug1', 'drug2']")
                    val_str = str(val).strip()
                    if val_str.startswith('[') and val_str.endswith(']'):
                        try:
                            import ast
                            items = ast.literal_eval(val_str)
                            extracted_items.extend([str(i).strip() for i in items if i])
                            continue
                        except:
                            pass
                    extracted_items.append(val_str)

        # Deduplicate while preserving order
        return list(dict.fromkeys(extracted_items))

    def get_recommendations(self, raw_disease_name):
        clean_input = self._clean_string(raw_disease_name)

        # 1. Fetch Description
        desc_matches = self.desc_df[self.desc_df['_clean_disease'] == clean_input]
        ignore_cols = ['_clean_disease', 'disease', 'diseases', 'unnamed: 0']
        desc_col = next((c for c in self.desc_df.columns if c.lower() not in ignore_cols), None)

        if not desc_matches.empty and desc_col:
            description = desc_matches[desc_col].values[0]
        else:
            description = "No description available."

        # 2. Extract multi-column values for other dataframes
        medications = self._extract_all_values(self.med_df, clean_input)
        precautions = self._extract_all_values(self.prec_df, clean_input)
        diets = self._extract_all_values(self.diet_df, clean_input)
        workouts = self._extract_all_values(self.work_df, clean_input)

        return {
            "Input Disease": raw_disease_name,
            "Matched Key": clean_input,
            "Description": description,
            "Medications": medications if medications else ["None listed"],
            "Precautions": precautions if precautions else ["None listed"],
            "Dietary Plan": diets if diets else ["None listed"],
            "Recommended Workouts": workouts if workouts else ["None listed"]
        }

In [ ]:
def print_recommendations(recommendation_result):
    if isinstance(recommendation_result, str):
        print(f"\n{recommendation_result}\n")
        return

    print(f"\n==========================================")
    print(f" CLINICAL RECOMMENDATIONS: {recommendation_result['Input Disease'].upper()}")
    print(f"==========================================\n")

    for section, content in recommendation_result.items():
        if section in ["Input Disease"]:
            continue

        print(f"--- {section} ---")
        if isinstance(content, list):
            for item in content:
                print(f"  • {item}")
        else:
            print(f"  {content}")
        print()

In [ ]:
# Initialize Recommender
recommender = ClinicalRecommenderSystem(
    description_df, medications_df, precautions_df, diets_df, workout_df
)

# Test 1: Valid Disease
result_valid = recommender.get_recommendations("Panic Disorder")
print_recommendations(result_valid)



 CLINICAL RECOMMENDATIONS: PANIC DISORDER

--- Matched Key ---
  panic disorder

--- Description ---
  Panic disorder is a mental health condition marked by sudden, unexpected panic attacks—intense periods of fear or discomfort—often accompanied by physical symptoms like chest pain, rapid heartbeat, shortness of breath, or dizziness.

--- Medications ---
  • SSRIs (e.g., Sertraline, Fluoxetine)
  • Benzodiazepines (e.g., Clonazepam, Alprazolam)
  • SNRIs (e.g., Venlafaxine)
  • Beta-blockers
  • Cognitive Behavioral Therapy (CBT)

--- Precautions ---
  • Practice deep breathing
  • Avoid caffeine
  • Follow therapy plan
  • Seek support from loved ones

--- Dietary Plan ---
  • Magnesium-rich foods (spinach, pumpkin seeds, almonds)
  • Omega-3 fatty acids (salmon, flaxseeds, walnuts)
  • Complex carbs (oats, quinoa)
  • Green tea (L-theanine)
  • Limit caffeine and sugar

--- Recommended Workouts ---
  • Deep breathing exercises: Calm your mind by focusing on slow, deep breaths
  • Yo